In [ ]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Downloaded to:", path)

In [ ]:
import os

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
img_dir_candidates = [root for root, dirs, files in os.walk(path) if any(f.lower().endswith(".jpg") for f in files)]

print("CSV files found:", csv_candidates)
print("Image directories found:", img_dir_candidates)

In [ ]:
import pandas as pd

metadata_path = next(p for p in csv_candidates if "metadata" in p.lower())
df = pd.read_csv(metadata_path)
print(df.shape)
print(df.columns.tolist())

class_counts = df["dx"].value_counts()
print()
print(class_counts)
print()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Imbalance ratio (largest/smallest class): {imbalance_ratio:.1f}x")
majority_baseline_acc = class_counts.max() / len(df)
print(f"'Always predict {class_counts.idxmax()}' baseline accuracy: {majority_baseline_acc:.1%}")

In [ ]:
def find_image_path(image_id):
    for img_dir in img_dir_candidates:
        candidate = os.path.join(img_dir, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None

df["image_path"] = df["image_id"].apply(find_image_path)

missing = df["image_path"].isna().sum()
print(f"Images not found: {missing} out of {len(df)}")
df = df.dropna(subset=["image_path"]).reset_index(drop=True)
print("Final shape:", df.shape)

In [ ]:
import numpy as np

DX_TO_FULLNAME = {
    "akiec": "Actinic keratoses and intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions",
}
DX_CODES = sorted(DX_TO_FULLNAME.keys())  # fixed order — this defines the model's output index order
CONDITIONS = [DX_TO_FULLNAME[c] for c in DX_CODES]

code_to_index = {code: i for i, code in enumerate(DX_CODES)}
df["label"] = df["dx"].map(code_to_index)

print("Class index mapping:")
for code, idx in code_to_index.items():
    print(f"  {idx}: {code} -> {DX_TO_FULLNAME[code]}")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print(f"Train: {len(train_df)}, Val: {len(val_df)}")